# Anotador TDAH · 02/04 · Backend `langchain` — salida estructurada con json_schema

Usa `ChatOllama` de LangChain con `with_structured_output(method="json_schema")`: el esquema JSON de la anotación (`AnotacionClinica`) se pasa a Ollama como `format`, y el modelo queda **obligado a nivel de decodificación** a emitir un JSON que cumpla el esquema. No depende de que el modelo "sepa" hacer tool calling.

Si la llamada estructurada falla, el backend cae a texto libre + parseo (mismo mecanismo que `directo`), y el campo `metodo` de la anotación registra cuál de las dos rutas se usó.

**El experimento de este cuaderno.** El dataset es longitudinal: 30 pacientes sintéticos con un reporte semanal del cuidador durante 24 semanas (720 entradas). La unidad clínica es **la anotación del reporte semanal**, así que la simulación va semana a semana: para cada semana elegida se anota la entrada de cada paciente `REPETICIONES` veces, y la fiabilidad se mide **dentro de cada semana** (¿anota el modelo lo mismo si le repito la misma semana?).

*Hipótesis previa:* Esperamos **fallo de formato ≈ 0** (el esquema se impone al decodificar). La pregunta interesante es si forzar el esquema también estabiliza el *contenido* (Jaccard de ítems) o solo la forma.

## 1 · Configuración

Todo lo que se puede tocar está en esta celda.

In [ ]:
import datetime as dt

import matplotlib.pyplot as plt
import pandas as pd

from anotador.config import Config
from anotador.instrumento import cargar_instrumento
from anotador.pipeline import anotar
from anotador.repositorio import ids_entradas_semana, semanas_disponibles
from anotador.simulacion import Rejilla, ejecutar
from anotador.analisis import (
    cargar_df,
    krippendorff_por_semana,
    resumen_por_semana,
)

pd.set_option("display.width", 200)

BACKEND = "langchain"
MODELO = "gemma4:e4b"     # pequeño para iterar; en Mercurio: gemma4:26b
TEMPERATURA = 0.7          # > 0 para que la variabilidad aflore
REPETICIONES = 5           # réplicas por entrada (mínimo razonable: 3)
SEMANAS = [1, 2]           # empezar con pocas; el dataset llega a la 24

instrumento = cargar_instrumento()
print(f"Instrumento: {instrumento['nombre']} ({len(instrumento['items'])} ítems)")
print(f"Semanas disponibles: {semanas_disponibles()}")

## 2 · Una anotación, para ver qué sale

Antes de simular nada, anotamos **una** entrada y miramos la salida completa. Aquí se ve el `metodo` que usó el backend y las 5 métricas de calidad.

In [ ]:
config = Config(backend=BACKEND, modelo=MODELO, temperature=TEMPERATURA)
id_ejemplo = ids_entradas_semana(SEMANAS[0])[0]

r = anotar(id_ejemplo, instrumento, config, persistir=False)
print(f"metodo={r.metodo} | formato_ok={r.formato_ok} | "
      f"metricas={r.n_metricas_ok}/5 | latencia={r.latencia_s:.1f}s")
r.anotacion

## 3 · Simulación semana a semana

Para cada semana: todas las entradas de esa semana (una por paciente) × `REPETICIONES`. Guardamos `inicio` antes de lanzar para poder analizar después **solo lo que produce esta simulación** (la tabla `anotacion` acumula todas las simulaciones históricas).

La celda estima primero el número de llamadas al modelo: con 30 pacientes, 2 semanas y 5 repeticiones son 300 llamadas.

In [ ]:
llamadas = sum(len(ids_entradas_semana(s)) for s in SEMANAS) * REPETICIONES
print(f"Llamadas al modelo: {llamadas}")

In [ ]:
inicio = dt.datetime.now()  # sello para filtrar el análisis

for semana in SEMANAS:
    ids = ids_entradas_semana(semana)
    print(f"— Semana {semana}: {len(ids)} pacientes × {REPETICIONES} repeticiones")
    ejecutar(
        Rejilla(
            entradas=ids,
            backends=[BACKEND],
            modelos=[MODELO],
            temperaturas=[TEMPERATURA],
            repeticiones=REPETICIONES,
        ),
        instrumento,
        verbose=False,
    )
print("Simulación completada.")

## 4 · Cargar SOLO esta simulación

`cargar_df(backend=..., desde=inicio)` filtra la tabla `anotacion` a lo que acabamos de ejecutar. Cada fila trae ya su `semana` y su `id_paciente`.

In [ ]:
df = cargar_df(backend=BACKEND, desde=inicio)
print(f"{len(df)} anotaciones de esta simulación")
df[["semana", "id_paciente", "repeticion", "nivel_alerta",
    "n_metricas_ok", "latencia_s"]].head(10)

## 5 · Fiabilidad por semana

- `jaccard_items` / `jaccard_escalas` (0–1): ¿elige el modelo los mismos ítems al repetir? 1.0 = siempre lo mismo.
- `acuerdo_nivel` (0–1): fracción de repeticiones que coincide con el nivel de alerta más votado.
- `alpha_nivel` (Krippendorff, ordinal): fiabilidad entre réplicas del nivel de alerta dentro de la semana (codificadores = repeticiones, unidades = pacientes). Referencia habitual: ≥ 0.80 fiable, 0.67–0.80 aceptable.
- `tasa_fallo_formato`: fracción de salidas que no se pudieron parsear/validar.

In [ ]:
resumen = resumen_por_semana(df)
display(resumen[["semana", "jaccard_items", "jaccard_escalas",
                 "acuerdo_nivel", "tasa_fallo_formato", "media_latencia"]])

alpha = krippendorff_por_semana(df)
if not alpha.empty:
    display(alpha[["semana", "alpha_nivel"]])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(resumen["semana"], resumen["jaccard_items"], "o-", label="Jaccard ítems")
ax[0].plot(resumen["semana"], resumen["acuerdo_nivel"], "s-", label="Acuerdo nivel")
if not alpha.empty:
    ax[0].plot(alpha["semana"], alpha["alpha_nivel"], "^--", label="Alpha Krippendorff")
ax[0].set_xlabel("semana"); ax[0].set_ylabel("fiabilidad")
ax[0].set_ylim(0, 1.05); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[0].set_title("Fiabilidad por semana — langchain")

ax[1].plot(resumen["semana"], resumen["media_latencia"], "d-", color="tab:red")
ax[1].set_xlabel("semana"); ax[1].set_ylabel("latencia media (s)")
ax[1].grid(alpha=0.3); ax[1].set_title("Coste por semana")

plt.tight_layout(); plt.show()

## 6 · Lectura del resultado

Para rellenar tras ejecutar (y llevar a la comparación del cuaderno 05):

| Pregunta | Respuesta |
|---|---|
| ¿Tasa de fallo de formato? | |
| ¿Jaccard de ítems medio? | |
| ¿Alpha del nivel por semana? | |
| ¿Latencia media por anotación? | |
| ¿La fiabilidad es estable entre semanas o depende de la semana? | |

La comparación entre los cuatro backends se hace en `05_comparacion_backends.ipynb`, que solo lee la base de datos (no lanza el modelo).